# Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Kiểm tra và cấu hình GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Thiết lập chế độ memory growth để TF không chiếm dụng hết VRAM ngay lập tức
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Hệ thống đã nhận diện {len(gpus)} GPU. Sẵn sàng huấn luyện mô hình!")
    except RuntimeError as e:
        print(f"Lỗi cấu hình GPU: {e}")
else:
    print("Không tìm thấy GPU. Hệ thống sẽ chạy bằng CPU (Tốc độ sẽ chậm hơn đáng kể).")

import warnings
warnings.filterwarnings("ignore")

# Load dataset
Đọc 2 file CSV của UNSW-NB15.

In [ ]:
import gc 
# Đường dẫn tới 2 file training và testing
train_path = '/kaggle/input/datasets/kevinnguyenuet/unsw-nb15/UNSW_NB15_training-set.csv'
test_path = '/kaggle/input/datasets/kevinnguyenuet/unsw-nb15/UNSW_NB15_testing-set.csv'

# Đọc 2 file 
print("Đang đọc file Training và Testing...")
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"KÍCH THƯỚC DỮ LIỆU:")
print(f"  - Tập Training: {train_df.shape}")
print(f"  - Tập Testing:  {test_df.shape}")

print("\n PHÂN BỐ LỚP (attack_cat) TRONG TẬP TRAINING:")
print(train_df['attack_cat'].value_counts())
display(train_df.head())
print("\n PHÂN BỐ LỚP (attack_cat) TRONG TẬP TESTING:")
print(test_df['attack_cat'].value_counts())
display(test_df.head())

# Data cleaning
Điền giá trị 0 cho các ô bị thiếu dữ liệu, xóa các mẫu lặp lại và các cột không có ý nghĩa cho mô hình

In [ ]:
# Hàm xử lý dữ liệu nhanh
def clean_dataset(df):
    print("Tổng số giá trị thiếu trên toàn bộ dataset trước khi xử lý:", df.isnull().sum().sum())
    # Điền số 0 vào giá trị thiếu
    df = df.fillna(0)
    # Loại bỏ trùng lặp
    df = df.drop_duplicates()
    # Loại bỏ cột không có ý nghĩa cho học máy
    df = df.drop(columns=['id', 'sttl', 'dttl', 'ct_state_ttl'], errors='ignore')
    return df

print("Đang làm sạch dữ liệu...")
train_df = clean_dataset(train_df)
test_df = clean_dataset(test_df)

print(f"Kích thước Train sau khi xử lý: {train_df.shape}")
print(f"Kích thước Test sau khi xử lý: {test_df.shape}")

# Chia dữ liệu (Tránh Data Leak)

In [ ]:
X_train_full = train_df.drop(columns=['label', 'attack_cat'])
y_train_full = train_df['label']
X_test = test_df.drop(columns=['label', 'attack_cat'])
y_test = test_df['label']

del train_df, test_df
gc.collect()

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full
)
del X_train_full, y_train_full
gc.collect()

print('X_train shape:', X_train.shape)
print('X_val shape:', X_val.shape)
print('X_test shape:', X_test.shape)
print('Train label distribution:', Counter(y_train))
print('Validation label distribution:', Counter(y_val))

# Scaling đồng thời one-hot
Sử dụng ColumnTransformer: MinMax cho cột số, OneHot cho cột hạng mục (`proto`, `service`, `state`).

In [ ]:
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline

categorical_cols = ['proto', 'service', 'state']
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

# Định nghĩa Pipeline cho cột số: Log1p -> MinMaxScaler
# np.log1p tính ln(1+x), giúp nén các giá trị cực lớn mà không sợ lỗi với số 0
numeric_transformer = Pipeline(steps=[
    ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')), # Thêm feature_names_out
    ('scaler', MinMaxScaler())
])

# Khởi tạo Preprocessor với OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ],
    verbose_feature_names_out=False
)

print("Đang tiền xử lý (Scale + OneHot)...")
# FIT và TRANSFORM trên tập Train
X_train_processed = preprocessor.fit_transform(X_train)
# Chỉ transform trên tập test và val
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Lấy lại tên cột
all_feature_names = preprocessor.get_feature_names_out()

# Khôi phục lại thành DataFrame
X_train_df = pd.DataFrame(X_train_processed, columns=all_feature_names, index=X_train.index)
X_test_df = pd.DataFrame(X_test_processed, columns=all_feature_names, index=X_test.index)
X_val_df = pd.DataFrame(X_val_processed, columns=all_feature_names, index=X_val.index)

# del X_train, X_test, X_val, X_train_processed, X_test_processed, X_val_processed# Giải phóng ram
# gc.collect()

print("Kích thước X_train sau tiền xử lý:", X_train_df.shape)
display(X_train_df.head())
print("Kích thước X_val sau tiền xử lý:", X_val_df.shape)
display(X_val_df.head())
print("Kích thước X_test sau tiền xử lý:", X_test_df.shape)
display(X_test_df.head())

# Liệt kê toàn bộ đặc trưng

In [ ]:
# --- Code liệt kê toàn bộ đặc trưng ---
print("\n" + "="*50)
print(f"DANH SÁCH TOÀN BỘ {len(all_feature_names)} ĐẶC TRƯNG:")
print("="*50)

# Liệt kê theo dạng danh sách có đánh số
for i, feature in enumerate(all_feature_names):
    print(f"{i+1:>3}. {feature}")

# Vẽ ma trận tương quan

In [ ]:
# VẼ BIỂU ĐỒ TƯƠNG QUAN SAU KHI ENCODING & SCALING
# Tại bước này, tập dữ liệu đã có rất nhiều cột do encoding.
# Biểu đồ này giúp thấy rõ mối quan hệ của từng đặc trưng cụ thể với nhãn 'label'.

# Tạo bản sao để tính toán (tránh ảnh hưởng đến X_train_df gốc)
# Chúng ta kết hợp X_train_df với y_train để xem tương quan với nhãn mục tiêu
temp_df = X_train_df.copy()
temp_df['target_label'] = y_train.values

# Tính toán ma trận tương quan
corr_matrix = temp_df.corr()

# Vì sau One-Hot số lượng cột rất lớn 
# Ta sẽ vẽ Heatmap tổng quát (không hiện số annot) để tránh rối mắt.
plt.figure(figsize=(25, 20))
sns.heatmap(corr_matrix, 
            annot=False, 
            cmap='RdBu_r', # Màu đỏ cho tương quan thuận, xanh cho tương quan nghịch
            center=0,
            linewidths=0.1)

plt.title('Ma trận tương quan sau khi One-Hot Encoding & Scaling', fontsize=18)
plt.show()

# Giải phóng bộ nhớ
del temp_df, corr_matrix
gc.collect()

# Training and validation

In [ ]:
# Khai báo biến layers từ Keras
layers = tf.keras.layers

# Định nghĩa các block của kiến trúc DuaNet
def plain_block(x, units: int, kernel_size: int, dropout: float, pool_stride: int, name: str):
    z = layers.BatchNormalization(name=f"{name}_bn_spatial")(x)
    z = layers.SeparableConv1D(filters=units, kernel_size=kernel_size, padding="same", activation="relu", name=f"{name}_dsc")(z)
    z = layers.MaxPooling1D(pool_size=2, strides=pool_stride, padding="same", name=f"{name}_mp")(z)
    z = layers.BatchNormalization(name=f"{name}_bn_temporal")(z)
    z = layers.GRU(units, return_sequences=True, activation="tanh", recurrent_activation="sigmoid", name=f"{name}_gru")(z)
    z = layers.Dropout(dropout, name=f"{name}_dropout")(z)
    return layers.TimeDistributed(layers.Dense(units, activation="linear"), name=f"{name}_linear")(z)

def dense_block(x, units: int, growth_rate: int, kernel_size: int, dropout: float, name: str):
    features = [x]
    for b in range(growth_rate):
        src = features[0] if len(features) == 1 else layers.Concatenate(name=f"{name}_concat_{b}")(features)
        out = plain_block(src, units, kernel_size, dropout, pool_stride=1, name=f"{name}_plain_{b + 1}")
        features.append(out)
    return layers.Concatenate(name=f"{name}_output")(features)

def transition_block(x, units: int, kernel_size: int, dropout: float, name: str):
    return plain_block(x, units, kernel_size, dropout, pool_stride=2, name=name)

def build_dualnet(n_features: int, learning_rate: float = 1e-3):
    # Cấu hình DuaNet: 3 dense blocks và có Attention
    n_dense_blocks = 3
    growth_rate = 4
    kernel_size = 10
    dropout = 0.2
    
    inputs = layers.Input(shape=(n_features,), name="feature_sequence")
    
    # Chuyển đổi dữ liệu từ 2D sang 3D để đưa vào Conv1D/GRU
    x = layers.Reshape((n_features, 1))(inputs)
    
    for idx in range(n_dense_blocks):
        x = dense_block(x, n_features, growth_rate, kernel_size, dropout, name=f"dense_{idx + 1}")
        if idx < n_dense_blocks - 1:
            x = transition_block(x, n_features, kernel_size, dropout, name=f"transition_{idx + 1}")
            
    # Self-attention đặc trưng của DuaNet
    x = layers.Attention(use_scale=False, name="self_attention")([x, x])
        
    x = layers.GlobalAveragePooling1D(name="global_average_pooling")(x)
    
    # Ouput cho bài toán phân loại nhị phân
    outputs = layers.Dense(1, activation="sigmoid", name="classifier")(x)
    
    model = tf.keras.Model(inputs, outputs, name="DualNet")
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
    )
    return model

# Khởi tạo chiến lược phân tán để chạy trên tất cả các GPU khả dụng
strategy = tf.distribute.MirroredStrategy()
print(f"Số lượng GPU đang được sử dụng để training: {strategy.num_replicas_in_sync}")

# Khởi tạo và compile model BÊN TRONG scope của strategy
with strategy.scope():
    model = build_dualnet(n_features=X_train_df.shape[1], learning_rate=0.001)

early_stopping = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)

# Điều chỉnh lại batch_size. 
# Khi chạy nhiều GPU, GLOBAL_BATCH_SIZE nên bằng batch_size trên mỗi GPU nhân với số lượng GPU.
BATCH_SIZE_PER_REPLICA = 256
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync

history = model.fit(
    X_train_df, y_train,
    validation_data=(X_val_df, y_val),
    epochs=60,
    batch_size=GLOBAL_BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)

# Vẽ đồ thị các độ đo đánh giá của mô hình

In [ ]:
# Lấy dữ liệu các chỉ số từ quá trình huấn luyện
hist = history.history

# Danh sách các metric đã được track trong hàm compile() của bạn
# Cấu trúc: (Tên metric tập train, Tên metric tập val, Tên hiển thị trên đồ thị)
metrics_to_plot = [
    ('accuracy', 'val_accuracy', 'Accuracy'),
    ('loss', 'val_loss', 'Loss'),
    ('precision', 'val_precision', 'Precision'),
    ('recall', 'val_recall', 'Recall')
]

# Vẽ từng biểu đồ riêng biệt cho mỗi chỉ số
for train_metric, val_metric, label_name in metrics_to_plot:
    plt.figure(figsize=(8, 5))
    
    # Vẽ đường cho tập Training và Validation
    plt.plot(hist[train_metric], label=f'Training {label_name}', color='#4C92C3') # Mã màu tương đồng ảnh
    plt.plot(hist[val_metric], label=f'Validation {label_name}', color='#FF9933')
    
    # Thiết lập nhãn và chú thích giống với image_1fa1a5.png
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel(label_name, fontsize=12)
    plt.legend(loc='upper left', fontsize=11)

    # THÊM ĐOẠN NÀY ĐỂ CỐ ĐỊNH TRỤC TUNG TỪ 0 ĐẾN 1
    if label_name != 'Loss':
        plt.ylim(0, 1)
        
    # Tùy chỉnh khung lưới (để tắt grid giống như trong ảnh mẫu)
    plt.grid(False)
    
    # Hiển thị biểu đồ
    plt.tight_layout()
    plt.show()

# Vẽ confusion matrix và đánh giá tập test

In [ ]:
start_time = time.time()
y_pred_prob = model.predict(X_test_df, batch_size=128)
inference_time_total = time.time() - start_time
inference_time_per_sample = inference_time_total / len(X_test_df)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Lấy các giá trị từ Confusion Matrix để tính DR và FAR
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

# Tính Detection Rate (DR) và False Alarm Rate (FAR)
detection_rate = tp / (tp + fn) if (tp + fn) > 0 else 0.0
false_alarm_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print(f'Accuracy: {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')
print(f'F1-score: {f1:.4f}')
print(f'Detection Rate (DR): {detection_rate:.4f}')
print(f'False Alarm Rate (FAR): {false_alarm_rate:.4f}')
print(f'Inference time: {inference_time_total:.4f}s for {len(X_test_df):,} samples')
print(f'Time per sample: {inference_time_per_sample * 1000:.4f} ms')

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues', cbar=False,
    xticklabels=['Normal (0)', 'Attack (1)'],
    yticklabels=['Normal (0)', 'Attack (1)']
)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('DNN Confusion Matrix')
plt.show()

# Lưu model

In [ ]:
model.save('unsw_nids_dnn_model.keras')
joblib.dump(preprocessor, 'unsw_preprocessor.joblib') 

print("Đã lưu Pipeline hoàn chỉnh cho PI 4 (UNSW)!")